# nanowhale 🐳 — 1B MoE Pretraining on Colab

**DeepSeek-V4 Mixture-of-Experts architecture at ~1.2B scale**

- ~400M active parameters per token (2-3x faster than dense 1B models)
- 256k context via curriculum training
- Reasoning-heavy dataset (code + math + logic + general knowledge)
- **Runs on free Colab H100 / A100 80GB** — no DeepSpeed needed

> Target: Beat Qwen3.5-0.8B with 2x inference speed via MoE sparsity

## Step 1: Setup — Install Dependencies & Clone Repo

In [ ]:
# Install everything (takes ~2 min)
!pip install -q torch transformers datasets safetensors pyyaml accelerate

# Clone the nanowhale repo
!git clone https://github.com/RemySkye/nanowhale
%cd nanowhale

## Step 2: Verify GPU (should be H100 or A100)

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## Step 3: Prepare Reasoning Dataset (~15-20 min)

Downloads and processes high-quality public datasets:
- `HuggingFaceFW/fineweb-2` (general English)
- `allenai/c4` (clean web text)
- `m-a-p/FineFineWeb` (curated high-quality)
- `openai/gsm8k` (math reasoning)
- `deepmind/code_contests` (competitive coding)
- `google-research-datasets/mbpp` (Python problems)

Long-context data is packed to 64k tokens for curriculum training.

In [ ]:
# Run data preparation (processes ~60-80k high-quality examples)
!python scripts/prepare_colab_data.py \
    --output data/processed/1b_moe_reasoning \
    --max_per_dataset 10000 \
    --pack_long_to 65536

## Step 4: Train 1B MoE Model

Training settings optimized for H100/A100 80GB:
- Batch size: 4 × 16 grad accum = effective 64
- BF16 mixed precision + torch.compile
- Curriculum context: 4k → 8k → 32k → 64k → 128k → 256k
- ~50k steps total

**Runtime estimate**: ~12-24 hours on H100 (50k steps)

In [ ]:
# Quick smoke test first (50 steps to verify everything works)
!python scripts/train_colab.py \
    --config configs/1b_moe_colab_256k.yaml \
    --output /content/checkpoints/smoke_test

# If the smoke test passes, run full training:
# !python scripts/train_colab.py \
#     --config configs/1b_moe_colab_256k.yaml \
#     --output /content/checkpoints/1b_moe_full

## Step 5: Full Training (uncomment when ready)

This cell runs the full 50k-step pretraining. Uncomment to run.

In [ ]:
# Full 50k-step training — uncomment to run
# !python scripts/train_colab.py \
#     --config configs/1b_moe_colab_256k.yaml \
#     --output /content/checkpoints/1b_moe_full

## Step 6: Save Model to Google Drive

Mounts Google Drive and copies the final checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = "/content/checkpoints/1b_moe_full/final"
dst = "/content/drive/MyDrive/nanowhale-1b-moe"
if os.path.exists(src):
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Model saved to Google Drive: {dst}")
else:
    print(f"No checkpoint found at {src} — run training first")

## Step 7: Quick Inference Test

Load the trained model and generate text.

In [ ]:
import torch
from transformers import PreTrainedTokenizerFast
from safetensors.torch import load_file

from configuration_deepseek_v4 import DeepseekV4Config
from modeling_deepseek_v4 import DeepseekV4ForCausalLM

# Load model
checkpoint = "/content/checkpoints/smoke_test/final"
config = DeepseekV4Config(
    vocab_size=128000, hidden_size=768, num_hidden_layers=16,
    num_attention_heads=16, num_key_value_heads=1, head_dim=128,
    qk_rope_head_dim=32, q_lora_rank=384, o_groups=4, o_lora_rank=192,
    moe_intermediate_size=2048, n_routed_experts=12, n_shared_experts=2,
    num_experts_per_tok=2, hc_mult=2, max_position_embeddings=4096,
)
model = DeepseekV4ForCausalLM(config).cuda()
state = load_file(f"{checkpoint}/model.safetensors")
model.load_state_dict(state, strict=False)

tokenizer = PreTrainedTokenizerFast.from_pretrained("tokenizer")

# Generate
prompt = "def fibonacci(n):"
input_ids = tokenizer.encode(prompt, return_tensors="pt").cuda()
with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=100, temperature=0.7, do_sample=True)
print(tokenizer.decode(output[0]))

---
## Custom Training Parameters

Edit these before running if needed:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `--config` | `configs/1b_moe_colab_256k.yaml` | Model + training config |
| `--output` | `/content/checkpoints/1b_moe_full` | Output directory |
| `--resume` | (none) | Resume from checkpoint |

**Hyperparams in `configs/1b_moe_colab_256k.yaml`**:
- `learning_rate`: 3e-4 (cosine schedule)
- `per_device_train_batch_size`: 4
- `gradient_accumulation_steps`: 16
- `max_steps`: 50,000
- Curriculum context: 4k → 256k over 25k steps